# 渐近理论的数值核对

配合 [theory_asymptotics.md](theory_asymptotics.md)。两项检验：

1. **收敛速度。** 命题 7：t(ν)、2 < ν < 4 时样本协方差比值的误差为 $T^{-(1-2/\nu)}$ 量级，极限是 (ν/2)-稳定分布，方差无穷。
   所以不能用方差的衰减检验速度（[bias_variance.ipynb](bias_variance.ipynb) 第 4.3 节的做法），要用分位数：极限连续时，四分位距（IQR）按同样的速度收敛。
   这里直接读取 bias_variance.ipynb 的缓存结果。
2. **置信区间覆盖率。** 由命题 4–6 的渐近方差构造三种 95% 区间，看实际覆盖率。

## 0 · 环境

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
RERUN = False
QUICK = os.environ.get("S09_QUICK") == "1"
RES = Path("results")


def cached(name, fn):
    path = RES / f"{name}{'_quick' if QUICK else ''}.csv.gz"
    if path.exists() and not RERUN:
        return pd.read_csv(path)
    df = fn()
    df.to_csv(path, index=False, float_format="%.6g")
    return df

In [2]:
import numpy as np, pandas as pd
from numpy.linalg import cholesky
from scipy.optimize import minimize
from scipy.stats import norm, t as student_t

ASSETS = ["沪深300", "中证500", "标普500", "国债", "黄金", "原油"]
N_ASSETS = 6
ANN_VOL = np.array([0.25, 0.28, 0.18, 0.03, 0.15, 0.32])
DAYS = 244
SIG = ANN_VOL / np.sqrt(DAYS)
CORR = np.array([
    [1.00, 0.85, 0.25, -0.10, 0.05, 0.20],
    [0.85, 1.00, 0.20, -0.10, 0.05, 0.20],
    [0.25, 0.20, 1.00, -0.20, 0.05, 0.30],
    [-0.10, -0.10, -0.20, 1.00, 0.15, -0.10],
    [0.05, 0.05, 0.05, 0.15, 1.00, 0.25],
    [0.20, 0.20, 0.30, -0.10, 0.25, 1.00],
])
SIGMA = np.outer(SIG, SIG) * CORR
L_CORR = cholesky(CORR)
UPPER = 0.40
CRASH_DIR = np.array([-3.0, -3.0, -2.5, 1.0, 0.5, -2.5])
_EK, _EK2 = 2.0, 5.0
P_CRASH = 0.01

def _crash_transform(p):
    m = p * _EK * CRASH_DIR
    cov = CORR + p * _EK2 * np.outer(CRASH_DIR, CRASH_DIR) - np.outer(m, m)
    return m, np.linalg.inv(cholesky(cov)).T @ L_CORR.T
_CRASH_M, _CRASH_A = _crash_transform(P_CRASH)

NUS = [3, 4, 5, 8, 15]
DGPS = ["高斯"] + [f"t({v})" for v in NUS] + ["崩盘混合"]
NU_OF = {f"t({v})": v for v in NUS}
QS = [0.95, 0.975, 0.99]

def draw_innov(kind, n, rng):
    z = rng.standard_normal((n, N_ASSETS)) @ L_CORR.T
    if kind == "高斯":
        return z
    if kind in NU_OF:
        nu = NU_OF[kind]
        return z * np.sqrt((nu - 2) / rng.chisquare(nu, n))[:, None]
    if kind == "崩盘混合":
        k = (rng.random(n) < P_CRASH) * (1 + rng.exponential(1.0, n))
        return (z + k[:, None] * CRASH_DIR - _CRASH_M) @ _CRASH_A
    raise ValueError(kind)

def k_es(kind, q):
    """单位方差下 ES 的常数 k_q；崩盘混合不是椭圆分布，返回 None。"""
    if kind == "高斯":
        return norm.pdf(norm.ppf(q)) / (1 - q)
    if kind in NU_OF:
        nu = NU_OF[kind]; tq = student_t.ppf(q, nu)
        return np.sqrt((nu - 2) / nu) * student_t.pdf(tq, nu) / (1 - q) * (nu + tq ** 2) / (nu - 1)
    return None

def es_sample(x, q, axis=0):
    """样本 ES（损失为正）：沿 axis 取最差 ⌈(1−q)n⌉ 个收益的平均损失。"""
    n = x.shape[axis]
    k = max(int(np.ceil((1 - q) * n)), 1)
    part = np.partition(x, k - 1, axis=axis)
    return -np.take(part, np.arange(k), axis=axis).mean(axis=axis)

# 固定组合：真实 Σ 下的最小方差（单资产上限 40%，不做空）与等权。
def min_var_capped(S, upper=UPPER):
    n = len(S)
    res = minimize(lambda w: w @ S @ w, np.full(n, 1 / n), jac=lambda w: 2 * S @ w,
                   bounds=[(0, upper)] * n, constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
                   method="SLSQP", options={"ftol": 1e-15, "maxiter": 500})
    return res.x

W_MIN = min_var_capped(SIGMA)
W_EW = np.full(N_ASSETS, 1 / N_ASSETS)
W_PAIR = np.column_stack([W_MIN, W_EW])            # 列：w*min、EW
SD = np.sqrt(np.einsum("ij,jk,ki->i", W_PAIR.T, SIGMA, W_PAIR))   # σ_w
RHO = (W_MIN @ SIGMA @ W_EW) / (SD[0] * SD[1])

# ── Tyler 形状估计量 ─────────────────────────────────────────────────────────
def tyler_shape(X, tol=1e-9, max_iter=500):
    """批量 Tyler M 估计量。X：(B, T, d)，已中心化。返回 (B, d, d)，tr = d，以及迭代次数。
    不动点迭代 V ← (d/T) Σ x xᵀ / (xᵀ V⁻¹ x)，每步归一化 tr(V) = d（Tyler 1987；Kent & Tyler 1988 证明了收敛）。"""
    B, T, d = X.shape
    V = np.broadcast_to(np.eye(d), (B, d, d)).copy()
    for it in range(1, max_iter + 1):
        q = np.einsum("bti,bij,btj->bt", X, np.linalg.inv(V), X)
        Vn = (d / T) * np.einsum("bti,btj->bij", X / q[..., None], X)
        Vn *= d / np.trace(Vn, axis1=1, axis2=2)[:, None, None]
        if np.max(np.abs(Vn - V)) < tol:
            return Vn, it
        V = Vn
    return V, max_iter


def ratio_from_shape(V):
    """形状矩阵 V（可带批次维）给出的 ES 比值 √(w1ᵀVw1 / w2ᵀVw2)。"""
    a = np.einsum("i,...ij,j->...", W_MIN, V, W_MIN)
    b = np.einsum("i,...ij,j->...", W_EW, V, W_EW)
    return np.sqrt(a / b)


def tyler_theory(kind, n=2_000_000, seed=2025):
    """Tyler 比值的概率极限（大样本上的 Tyler 估计）与椭圆族的渐近方差 (d+2)/d·(1−ρ²)。"""
    if kind in NU_OF or kind == "高斯":
        lim = SD[0] / SD[1]                       # 椭圆族：Tyler 的极限与 Σ 的形状相同
        avar_rel = (N_ASSETS + 2) / N_ASSETS * (1 - RHO ** 2)
    else:
        Z = draw_innov(kind, n, np.random.default_rng(seed)) * SIG
        V, _ = tyler_shape(Z[None], tol=1e-12)
        lim = ratio_from_shape(V[0])
        avar_rel = np.nan                         # 非椭圆：没有闭式，用模拟值
    return lim, avar_rel


def run_cell_tyler(kind, T, reps, seed, batch=100):
    """与 run_cell 使用相同的随机数流（同一 seed、同一 batch），所以样本与高斯插件、HS 完全配对。"""
    rng = np.random.default_rng(seed)
    out = []
    for b0 in range(0, reps, batch):
        B = min(batch, reps - b0)
        R = draw_innov(kind, B * T, rng).reshape(B, T, N_ASSETS) * SIG
        V, it = tyler_shape(R - R.mean(axis=1, keepdims=True))
        out.append(pd.DataFrame({"dgp": kind, "T": T, "rep": np.arange(b0, b0 + B),
                                 "tyler_ratio": ratio_from_shape(V), "iters": it}))
    return pd.concat(out, ignore_index=True)

## 1 · 收敛速度：IQR 的双对数斜率

$\log\mathrm{IQR}(\hat\theta/\theta-1)$ 对 $\log T$（T = 125…4000）回归的斜率。预测：√T 收敛为 −0.5；
样本协方差比值在 2 < ν < 4 时为 $-(1-2/\nu)$；ν = 4 时为 $\sqrt{\log T/T}$，在这个 T 范围内的平均斜率约为 $-0.5+1/(2\overline{\log T})\approx-0.42$。

In [3]:
S = pd.read_csv(RES / "expD_biasvar.csv.gz").merge(pd.read_csv(RES / "expE_tyler.csv.gz"), on=["dgp", "T", "rep"])
S["ty_rat"] = S["tyler_ratio"] / (SD[0] / SD[1]) - 1          # 椭圆族真值 = σ1/σ2
ELL = [k for k in DGPS if k != "崩盘混合"]


def pred_g(kind):
    nu = NU_OF.get(kind, np.inf)
    if nu < 4:
        return -(1 - 2 / nu)
    if nu == 4:
        return -0.5 + 1 / (2 * np.mean(np.log(sorted(S["T"].unique()))))
    return -0.5


rows = []
for kind in ELL:
    r = {"dgp": kind, "预测（样本协方差）": pred_g(kind)}
    for col, lab in [("g_rat_950", "样本协方差"), ("ty_rat", "Tyler"), ("hs_rat_950", "HS")]:
        g = S[S.dgp == kind].groupby("T")[col]
        iqr = g.quantile(.75) - g.quantile(.25)
        r[lab] = np.polyfit(np.log(iqr.index.to_numpy(float)), np.log(iqr.to_numpy()), 1)[0]
    rows.append(r)
rate = pd.DataFrame(rows).set_index("dgp")
rate.style.format("{:.3f}")

,预测（样本协方差）,样本协方差,Tyler,HS
dgp,,,,
高斯,-0.500,-0.504,-0.494,-0.503
t(3),-0.333,-0.332,-0.501,-0.442
t(4),-0.424,-0.431,-0.514,-0.477
t(5),-0.500,-0.459,-0.507,-0.473
t(8),-0.500,-0.505,-0.507,-0.490
t(15),-0.500,-0.501,-0.499,-0.490


## 2 · 置信区间覆盖率

三种 95% 区间，都是 $\hat\theta\,(1\pm1.96\sqrt{\hat V/T})$：
- **样本协方差**：$\hat V=\tfrac14\widehat{\mathrm{Var}}(u_1^2-u_2^2)$，$u_j$ 为组合 $j$ 的标准化收益（命题 4，只需四阶矩，不需要椭圆假设）。
- **Tyler**：$\hat V=\frac{d+2}{d}(1-\hat\rho^2)$，$\hat\rho$ 由 Tyler 形状矩阵算出（命题 5，依赖椭圆假设）。
- **HS**：$\hat V=\widehat{\mathrm{Var}}(\mathrm{IF}_1/\widehat{\mathrm{ES}}_1-\mathrm{IF}_2/\widehat{\mathrm{ES}}_2)$，影响函数用经验 VaR 代入（命题 6）。

目标都是真实 ES 比值 θ（q = 0.95）。崩盘混合下前两种估计量不相合，覆盖率应趋于 0。每格 2000 次重复。

In [4]:
Q = 0.95
Z = norm.ppf(0.975)
COV_DGPS = ["高斯", "t(8)", "t(5)", "t(4)", "t(3)", "崩盘混合"]
COV_T = [250, 1000] if QUICK else [250, 1000, 4000]
COV_REPS = 100 if QUICK else 2000
THETA = pd.read_csv(RES / "expD_theory.csv.gz").set_index(["dgp", "q"])["theta"]


def coverage_cell(kind, T, reps, seed, batch=100):
    rng = np.random.default_rng(seed)
    th = THETA[(kind, Q)]
    k = int(np.ceil((1 - Q) * T))
    out = []
    for b0 in range(0, reps, batch):
        B = min(batch, reps - b0)
        R = draw_innov(kind, B * T, rng).reshape(B, T, N_ASSETS) * SIG
        Rc = R - R.mean(axis=1, keepdims=True)
        P = Rc @ W_PAIR                                            # 中心化后的组合收益
        sd = np.sqrt((P ** 2).sum(axis=1) / (T - 1))
        # 样本协方差
        u2 = (P / sd[:, None, :]) ** 2
        th_g = sd[:, 0] / sd[:, 1]
        v_g = 0.25 * np.var(u2[..., 0] - u2[..., 1], axis=1, ddof=1)
        # Tyler
        V, _ = tyler_shape(Rc)
        th_t = ratio_from_shape(V)
        c = np.einsum("i,bij,j->b", W_MIN, V, W_EW)
        a = np.einsum("i,bij,j->b", W_MIN, V, W_MIN)
        b = np.einsum("i,bij,j->b", W_EW, V, W_EW)
        v_t = (N_ASSETS + 2) / N_ASSETS * (1 - c ** 2 / (a * b))
        # HS：损失 = −收益（用未中心化的收益，与 es_sample 一致）
        L = -(R @ W_PAIR)
        var_hat = -np.partition(-L, k - 1, axis=1)[:, k - 1, :]    # 第 k 大的损失
        es = es_sample(-L, Q, axis=1)
        ifn = var_hat[:, None, :] + np.maximum(L - var_hat[:, None, :], 0) / (1 - Q)
        rel = ifn / es[:, None, :]
        th_h = es[:, 0] / es[:, 1]
        v_h = np.var(rel[..., 0] - rel[..., 1], axis=1, ddof=1)
        rec = {"dgp": kind, "T": T, "rep": np.arange(b0, b0 + B)}
        for lab, est, v in [("样本协方差", th_g, v_g), ("Tyler", th_t, v_t), ("HS", th_h, v_h)]:
            half = Z * np.sqrt(v / T)
            rec[f"cover_{lab}"] = (est * (1 - half) <= th) & (th <= est * (1 + half))
            rec[f"width_{lab}"] = 2 * half
        out.append(pd.DataFrame(rec))
    return pd.concat(out, ignore_index=True)


def run_coverage():
    from joblib import Parallel, delayed
    jobs = [(k, T, [7, i, j]) for i, k in enumerate(COV_DGPS) for j, T in enumerate(COV_T)]
    return pd.concat(Parallel(n_jobs=-1)(delayed(coverage_cell)(k, T, COV_REPS, s) for k, T, s in jobs), ignore_index=True)


COV = cached("expF_coverage", run_coverage)
labs = ["样本协方差", "Tyler", "HS"]
cov = COV.groupby(["dgp", "T"], sort=False)[[f"cover_{l}" for l in labs]].mean() * 100
cov.columns = labs
print("覆盖率（%，名义 95%；2000 次重复时蒙特卡洛标准误约 0.5 个百分点）")
cov.loc[COV_DGPS].style.format("{:.1f}")

覆盖率（%，名义 95%；2000 次重复时蒙特卡洛标准误约 0.5 个百分点）


In [5]:
wid = COV.groupby(["dgp", "T"], sort=False)[[f"width_{l}" for l in labs]].median() * 100
wid.columns = labs
print("区间宽度的中位数（相对 θ 的 %）")
wid.loc[COV_DGPS].style.format("{:.2f}")

区间宽度的中位数（相对 θ 的 %）
